In [ ]:
using Pkg
Pkg.activate(".")

using CairoMakie
using HDF5
using Statistics
using LaTeXStrings

include("plotting_boilerplate.jl")
include("multiplier_scaling.jl")

In [ ]:
datadir = "../scripts/utils/0_random_variation/data"
data = load_sweep_data(datadir)
println("Loaded data for nqubits: $(sort(collect(keys(data))))")

In [ ]:
# Plot 1: Best accuracy vs multiplier, one curve per pzero (one figure per nqubit)
for nqubit in sort(collect(keys(data)))
    nq_data = data[nqubit]
    nstates, pzeros, multipliers = get_multipliers_and_pzeros(nq_data, nqubit)
    colors = cgrad(:roma, length(pzeros), categorical=true)

    fig = Figure(size=(700, 450))
    ax = Axis(fig[1, 1],
        xlabel=L"N_{\mathrm{state}} / N_{\mathrm{state},0}",
        ylabel="Best accuracy",
        title=L"n_{\mathrm{qubit}} = %$(nqubit)",
    )

    for (pidx, pzero) in enumerate(pzeros)
        meds, q1s, q3s, mults_valid = Float64[], Float64[], Float64[], Float64[]
        for (ns, mult) in zip(nstates, multipliers)
            key = (ns, pzero)
            haskey(nq_data, key) || continue
            vals = nq_data[key]
            push!(meds, median(vals))
            push!(q1s, quantile(vals, 0.25))
            push!(q3s, quantile(vals, 0.75))
            push!(mults_valid, mult)
        end
        lines!(ax, mults_valid, meds, color=colors[pidx],
               label=L"p_{\mathrm{zero}} = %$(pzero)")
        band!(ax, mults_valid, q1s, q3s, color=(colors[pidx], 0.2))
    end

    fig[1, 2] = Legend(fig, ax)
    display(fig)
end

In [ ]:
# Plot 2: Best accuracy vs multiplier, one curve per nqubit (one figure per pzero)
all_pzeros = sort(unique(vcat([collect(unique([k[2] for k in keys(nq)])) for nq in values(data)]...)))
nqubits = sort(collect(keys(data)))
colors = cgrad(:roma, length(nqubits), categorical=true)

for pzero in all_pzeros
    fig = Figure(size=(700, 450))
    ax = Axis(fig[1, 1],
        xlabel=L"N_{\mathrm{state}} / N_{\mathrm{state},0}",
        ylabel="Best accuracy",
        title=L"p_{\mathrm{zero}} = %$(pzero)",
    )

    for (nidx, nqubit) in enumerate(nqubits)
        nq_data = data[nqubit]
        nstates, _, multipliers = get_multipliers_and_pzeros(nq_data, nqubit)
        meds, q1s, q3s, mults_valid = Float64[], Float64[], Float64[], Float64[]
        for (ns, mult) in zip(nstates, multipliers)
            key = (ns, pzero)
            haskey(nq_data, key) || continue
            vals = nq_data[key]
            push!(meds, median(vals))
            push!(q1s, quantile(vals, 0.25))
            push!(q3s, quantile(vals, 0.75))
            push!(mults_valid, mult)
        end
        length(meds) == 0 && continue
        lines!(ax, mults_valid, meds, color=colors[nidx],
               label=L"n_{\mathrm{qubit}} = %$(nqubit)")
        band!(ax, mults_valid, q1s, q3s, color=(colors[nidx], 0.2))
    end

    fig[1, 2] = Legend(fig, ax)
    display(fig)
end

In [ ]:
# Plot 3: Heatmap multiplier x pzero (one figure per nqubit)
for nqubit in sort(collect(keys(data)))
    nq_data = data[nqubit]
    nstates, pzeros, multipliers = get_multipliers_and_pzeros(nq_data, nqubit)

    med_matrix = fill(NaN, length(multipliers), length(pzeros))
    for (midx, ns) in enumerate(nstates)
        for (pidx, pzero) in enumerate(pzeros)
            key = (ns, pzero)
            haskey(nq_data, key) || continue
            med_matrix[midx, pidx] = median(nq_data[key])
        end
    end

    fig = Figure(size=(600, 450))
    ax = Axis(fig[1, 1],
        xlabel=L"N_{\mathrm{state}} / N_{\mathrm{state},0}",
        ylabel=L"p_{\mathrm{zero}}",
        title=L"n_{\mathrm{qubit}} = %$(nqubit)",
    )

    hm = heatmap!(ax, multipliers, pzeros, med_matrix, colormap=:roma)
    Colorbar(fig[1, 2], hm, label="Median best accuracy")
    display(fig)
end